## Conexión y utilidades

In [31]:
from pymongo import MongoClient
import gridfs
from bson import ObjectId

client = MongoClient("mongodb://localhost:27017/")
db = client['trafico']
fs = gridfs.GridFS(db)
videos_col = db['Videos']
frames_col = db['VideosFrames']
videos_reconstruidos_col = db['VideosReconstruidos']

## Listar todos los videos y sus IDs

In [32]:
for video in videos_col.find():
    print("video_id:", video["_id"], "| nombre:", video["nombre"], "| fecha:", video["fecha_subida"])

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6847f9ebf449f0965f13d621, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

##  Consulta frames de un video específico

In [30]:
video_id = ObjectId("6847bb810270edd78fabb649")
for frame in frames_col.find({"video_id": video_id}).sort("frame_num", 1):
    print("frame_num:", frame["frame_num"], "| gridfs_id:", frame.get("gridfs_id"), "| metricas:", frame.get("metricas"))

frame_num: 0 | gridfs_id: 6847bb820270edd78fabb64a | metricas: {'total_vehicles': 2, 'by_class': {'car': 2}, 'average_confidence': 0.8837334215641022, 'timestamp': datetime.datetime(2025, 6, 9, 22, 58, 42, 464000), 'frame_num': 0, 'video_id': ObjectId('6847bb810270edd78fabb649'), 'waiting_time': 4, 'traffic_density': 0.2, 'hour': 22, 'day_of_week': 0, 'weather': 'cloudy'}
frame_num: 1 | gridfs_id: 6847bb820270edd78fabb64e | metricas: {'total_vehicles': 2, 'by_class': {'car': 2}, 'average_confidence': 0.9280883371829987, 'timestamp': datetime.datetime(2025, 6, 9, 22, 58, 42, 810000), 'frame_num': 1, 'video_id': ObjectId('6847bb810270edd78fabb649'), 'waiting_time': 4, 'traffic_density': 0.2, 'hour': 22, 'day_of_week': 0, 'weather': 'sunny'}
frame_num: 2 | gridfs_id: 6847bb830270edd78fabb652 | metricas: {'total_vehicles': 3, 'by_class': {'car': 3}, 'average_confidence': 0.904593308766683, 'timestamp': datetime.datetime(2025, 6, 9, 22, 58, 43, 123000), 'frame_num': 2, 'video_id': ObjectId(

## Limpiar datos de prueba o antiguos

In [21]:
# Eliminar todos los videos y frames marcados como test
videos_col.delete_many({"es_test": True})
frames_col.delete_many({"es_test": True})

# Eliminar todos los archivos de GridFS marcados como test (si guardaste ese campo)
for frame in frames_col.find({"es_test": True}):
    if "gridfs_id" in frame:
        fs.delete(frame["gridfs_id"])
print("Datos de prueba eliminados.")

Datos de prueba eliminados.


## Eliminar todos los archivos de GridFS (¡CUIDADO!

In [22]:
for file in fs.find():
    fs.delete(file._id)
print("Todos los archivos de GridFS eliminados.")

Todos los archivos de GridFS eliminados.


## Consultar frames con más de X autos detectados

In [23]:
for frame in frames_col.find({"metricas.by_class.car": {"$gt": 10}}):
    print("frame_num:", frame["frame_num"], "| autos:", frame["metricas"]["by_class"]["car"])

## Descargar y visualizar una imagen desde GridFS

In [24]:
import cv2
import numpy as np
from IPython.display import Image, display

frame = frames_col.find_one({"video_id": video_id, "frame_num": 0})  # Cambia el frame_num si quieres otro
if frame and "gridfs_id" in frame:
    img_bytes = fs.get(frame["gridfs_id"]).read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    cv2.imwrite("frame_temp.jpg", img)
    display(Image(filename="frame_temp.jpg"))
else:
    print("No se encontró el frame o gridfs_id.")

No se encontró el frame o gridfs_id.


## Estadísticas rápidas

In [25]:
print("Total de videos:", videos_col.count_documents({}))
print("Total de frames:", frames_col.count_documents({}))
print("Total de videos reconstruidos:", videos_reconstruidos_col.count_documents({}))

Total de videos: 1
Total de frames: 11
Total de videos reconstruidos: 1


## Eliminar TODOS los videos, frames y videos reconstruidos

In [26]:
videos_col.delete_many({})
frames_col.delete_many({})
videos_reconstruidos_col.delete_many({})

for file in fs.find():
    fs.delete(file._id)

print("Todos los datos y archivos de GridFS han sido eliminados!!!")

Todos los datos y archivos de GridFS han sido eliminados!!!
